# Data Cleaning and Preparation

This notebook prepares the original IITA tomato endline dataset for exploratory data analysis and machine learning.

The cleaning process focuses on the irrigated production observations while preserving original variable names, category labels and valid missing values.

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
df = pd.read_csv('tomato_endline-data.csv')

print("Original dataset shape:", df.shape)
display(df.head())

C:\Users\HP\AppData\Local\Temp\ipykernel_29556\443629184.py:1: DtypeWarning: Columns (62: srcPurchCrate, 64: supplrCrate, 201: placeWholeSale1_RF, 224: strgeLossUnit1_RF, 284: placeWholeSale2_RF, 307: strgeLossUnit2_RF, 390: strgeLossUnit3_RF, 393: trnsptLossUnit3_RF, 450: placeWholeSale4_RF, 473: strgeLossUnit4_RF, 476: trnsptLossUnit4_RF, 488: methdAcqCrate5_RF, 490: methdArrngCrateTrnsctn5_RF, 492: collectnPointCrates5_RF, 495: meansTrnsptCrateToFarm5_RF, 514: mnthSale5_RF, 515: weekSale5_RF, 516: qualitySale5_RF, 527: buyr5_RF, 529: mthdChooseWhoSell5_RF, 531: methdArrngTransctn5_RF, 533: placeHrvstSale5_RF, 537: mthdChooseWhrSell5_RF, 540: incurTrnsptCost5_RF, 542: methdTransptBulkTomato5_RF, 545: sellContract5_RF, 548: sellCollectvly5_RF, 549: hvMrktPrcInfoB4Sale5_RF, 550: mjrSrcMktPrcinfo5_RF, 552: decidrMktPrc5_RF, 869: trnsptLossUnit1_IRG, 1032: strgeLossUnit3_IRG, 1115: strgeLossUnit4_IRG, 1130: methdAcqCrate5_IRG, 1156: mnthSale5_IRG, 1157: weekSale5_IRG, 1158: qualitySale5_

Original dataset shape: (928, 1949)


,ID,hh_uuid,respGendr,respAge,respMStatus,othRespMStatus,respHHStructure,othrespHHStructure,respEduLev,respHHSize,...,copnStratgy_14,copnStratgy_15,copnStratgy_16,copnStratgy_17,copnStratgy_18,copnStratgy_19,copnStratgy_99,othcopnStratgy,hhRcoverCompletly,duratnHHRcvry
0,1,WVJT8O,Male,60,Married,,Male-headed polygamous household,,Religious (Koran) school,16,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,,No,NaN
1,2,RX6O9H,Male,45,Married,,Male-headed polygamous household,,Primary education,11,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,,No,NaN
2,3,JS4J0W,Male,56,Married,,Male-headed monogamous household,,Religious (Koran) school,13,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,,No,NaN
3,4,2EZMN3,Male,35,Married,,Male-headed polygamous household,,Religious (Koran) school,11,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,,No,NaN
4,5,2I3KYJ,Male,40,Married,,Male-headed polygamous household,,Religious (Koran) school,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN


In [6]:
# Identify irrigated production observations
irg_loss_columns = [
    'farmLoss_kg_IRG',
    'durngHrvstLoss_kg_IRG',
    'storgeLoss_kg_IRG',
    'trnsptLoss_kg_IRG'
]

# Select observations with at least one recorded irrigated loss value
irg_df = df.loc[
    df[irg_loss_columns].notna().any(axis=1)
].copy()

cleaned_df = irg_df.copy()

print("Irrigated dataset shape:", cleaned_df.shape)

Irrigated dataset shape: (877, 1949)


In [7]:
print("Dataset information:")
cleaned_df.info()

print("\nDescriptive statistics:")
display(cleaned_df.describe(include='all').T)

print("\nMissing values:")
print(cleaned_df.isna().sum().sum())

print("\nDuplicate rows:")
print(cleaned_df.duplicated().sum())

Dataset information:
<class 'pandas.DataFrame'>
Index: 877 entries, 0 to 927
Columns: 1949 entries, ID to duratnHHRcvry
dtypes: float64(864), int64(93), str(992)
memory usage: 13.0 MB

Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,877.0,NaN,NaN,NaN,466.054732,267.344777,1.0,232.0,466.0,698.0,928.0
hh_uuid,877,877,WVJT8O,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
respGendr,877,2,Male,853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
respAge,877.0,NaN,NaN,NaN,44.659065,11.132097,20.0,35.0,45.0,52.0,80.0
respMStatus,877,4,Married,853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
copnStratgy_19,733.0,NaN,NaN,NaN,0.196453,0.397586,0.0,0.0,0.0,0.0,1.0
copnStratgy_99,733.0,NaN,NaN,NaN,0.020464,0.141677,0.0,0.0,0.0,0.0,1.0
othcopnStratgy,877,14,,862,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hhRcoverCompletly,733,2,Yes,480,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing values:
883665

Duplicate rows:
0


In [8]:
# Remove accidental spaces around column names.
cleaned_df.columns = cleaned_df.columns.str.strip()

# Treat empty strings and whitespace-only strings as missing values.
cleaned_df = cleaned_df.replace(r'^\s*$', np.nan, regex=True)

print("Blank values standardised.")
print("Missing values:", cleaned_df.isna().sum().sum())

Blank values standardised.
Missing values: 1246476


In [9]:
duplicates_before = cleaned_df.duplicated().sum()

cleaned_df = cleaned_df.drop_duplicates().copy()

print("Duplicate rows found:", duplicates_before)
print("Dataset shape after duplicate check:", cleaned_df.shape)

Duplicate rows found: 0
Dataset shape after duplicate check: (877, 1949)


In [12]:
# Farmer and household characteristics
demographic_columns = [
    'ID', 'hh_uuid',
    'respGendr', 'respAge', 'respMStatus',
    'respEduLev', 'respHHSize', 'numAdultHH',
    'respPryOcc', 'respSecOcc'
]

# Farm characteristics
farm_columns = [
    'tomProdExp', 'prodSystm',
    'dst2MajrMarket', 'dstHme2Farm',
    'totSizFarm_23', 'totSizUnitFarm_23',
    'tomatoFarm_23', 'tomatoFarmUnit_23'
]

# Irrigated production and loss information
production_columns = [
    'totProd_IRG', 'unitProd_IRG',
    'totConsm_IRG', 'totSold_IRG',
    'farmLoss_kg_IRG', 'farmLossUnit_kg_IRG',
    'durngHrvstLoss_kg_IRG', 'durngHrvstLossUnit_kg_IRG',
    'storgeLoss_kg_IRG', 'storgeLossUnit_kg_IRG',
    'trnsptLoss_kg_IRG', 'trnsptLossUnit_kg_IRG'
]

# Storage and packaging practices
storage_columns = [
    'evrUsdPlstcCrate',
    'usePlstcCrateHarvest_IRG',
    'numPlstcCrateHarvest_IRG',
    'rcvTRaininStorgePrctc',
    'rcvTRaininPackgn',
    'hvStorgePlace',
    'mstCommnStorgPrac',
    'lngthStorgPerdBfrSale',
    'practcPrecooln',
    'cleanB4Marktn',
    'sortGradeB4Marktn',
    'useColdRmStrge',
    'awrZeroEnrgy',
    'usdZeroEnrgy'
]

# Agricultural practices
agricultural_columns = [
    'applyFert',
    'applyPestcde',
    'freqSprayPestcd',
    'mainSrcIrrigatn',
    'rcvIPMtrainig',
    'adoptIPM'
]

# Combine the selected variables
selected_columns = (
    demographic_columns +
    farm_columns +
    production_columns +
    storage_columns +
    agricultural_columns
)

# Retain only columns that actually exist
selected_columns = [
    col for col in selected_columns
    if col in cleaned_df.columns
]

eda_df = cleaned_df[selected_columns].copy()

print("Original irrigated dataset:", cleaned_df.shape)
print("EDA dataset:", eda_df.shape)
print("Selected columns:", len(selected_columns))

display(eda_df.head())

Original irrigated dataset: (877, 1949)
EDA dataset: (877, 50)
Selected columns: 50


,ID,hh_uuid,respGendr,respAge,respMStatus,respEduLev,respHHSize,numAdultHH,respPryOcc,respSecOcc,...,sortGradeB4Marktn,useColdRmStrge,awrZeroEnrgy,usdZeroEnrgy,applyFert,applyPestcde,freqSprayPestcd,mainSrcIrrigatn,rcvIPMtrainig,adoptIPM
0,1,WVJT8O,Male,60,Married,Religious (Koran) school,16,10,Farming,Agriculture trading,...,Yes,No,No,NaN,Yes,Yes,8.0,1,NaN,NaN
1,2,RX6O9H,Male,45,Married,Primary education,11,3,Farming,Agriculture trading,...,Yes,Yes,No,NaN,Yes,Yes,8.0,1,NaN,NaN
2,3,JS4J0W,Male,56,Married,Religious (Koran) school,13,9,Farming,Agriculture trading,...,Yes,No,No,NaN,Yes,Yes,5.0,1,NaN,NaN
3,4,2EZMN3,Male,35,Married,Religious (Koran) school,11,4,Farming,Other (specify),...,Yes,No,No,NaN,Yes,Yes,5.0,1,NaN,NaN
4,5,2I3KYJ,Male,40,Married,Religious (Koran) school,11,6,Farming,Agriculture trading,...,Yes,No,No,NaN,Yes,Yes,15.0,1,NaN,NaN


In [13]:
print("Missing values per column:")
display(
    eda_df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame('Missing Values')
)

print("\nData types:")
display(eda_df.dtypes.to_frame('Data Type'))

print("\nDescriptive statistics:")
display(eda_df.describe(include='all').T)

Missing values per column:


,Missing Values
numPlstcCrateHarvest_IRG,876
storgeLossUnit_kg_IRG,875
trnsptLossUnit_kg_IRG,862
usdZeroEnrgy,722
usePlstcCrateHarvest_IRG,625
adoptIPM,368
rcvIPMtrainig,368
respSecOcc,291
durngHrvstLossUnit_kg_IRG,237
evrUsdPlstcCrate,234



Data types:


,Data Type
ID,int64
hh_uuid,str
respGendr,str
respAge,int64
respMStatus,str
respEduLev,str
respHHSize,int64
numAdultHH,int64
respPryOcc,str
respSecOcc,str



Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,877.0,NaN,NaN,NaN,466.054732,267.344777,1.0,232.0,466.0,698.0,928.0
hh_uuid,877,877,WVJT8O,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
respGendr,877,2,Male,853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
respAge,877.0,NaN,NaN,NaN,44.659065,11.132097,20.0,35.0,45.0,52.0,80.0
respMStatus,877,4,Married,853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
respEduLev,877,8,Primary education,301,NaN,NaN,NaN,NaN,NaN,NaN,NaN
respHHSize,877.0,NaN,NaN,NaN,12.126568,6.108134,1.0,8.0,11.0,15.0,40.0
numAdultHH,877.0,NaN,NaN,NaN,4.706956,2.742713,0.0,3.0,4.0,6.0,22.0
respPryOcc,877,5,Farming,809,NaN,NaN,NaN,NaN,NaN,NaN,NaN
respSecOcc,586,6,Self-employed,201,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
print("FINAL CLEANED DATASET")
print("---------------------")
print("Shape:", cleaned_df.shape)
print("Missing values:", cleaned_df.isna().sum().sum())
print("Duplicate rows:", cleaned_df.duplicated().sum())

display(cleaned_df.head())

FINAL CLEANED DATASET
---------------------
Shape: (877, 1949)
Missing values: 1246476
Duplicate rows: 0


,ID,hh_uuid,respGendr,respAge,respMStatus,othRespMStatus,respHHStructure,othrespHHStructure,respEduLev,respHHSize,...,copnStratgy_14,copnStratgy_15,copnStratgy_16,copnStratgy_17,copnStratgy_18,copnStratgy_19,copnStratgy_99,othcopnStratgy,hhRcoverCompletly,duratnHHRcvry
0,1,WVJT8O,Male,60,Married,NaN,Male-headed polygamous household,NaN,Religious (Koran) school,16,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,No,NaN
1,2,RX6O9H,Male,45,Married,NaN,Male-headed polygamous household,NaN,Primary education,11,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,NaN,No,NaN
2,3,JS4J0W,Male,56,Married,NaN,Male-headed monogamous household,NaN,Religious (Koran) school,13,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,NaN,No,NaN
3,4,2EZMN3,Male,35,Married,NaN,Male-headed polygamous household,NaN,Religious (Koran) school,11,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,NaN,No,NaN
4,5,2I3KYJ,Male,40,Married,NaN,Male-headed polygamous household,NaN,Religious (Koran) school,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
eda_df.to_csv(
    '../data/processed/tomato_farm_loss_eda.csv',
    index=False
)

print("EDA dataset exported successfully.")
print("Shape:", eda_df.shape)

EDA dataset exported successfully.
Shape: (877, 50)
